# Q2. Unsupervised Learning — Customer Segmentation using K-Means

**Objective:** Perform customer segmentation using K-Means clustering and visualise the clusters using PCA.

**Dataset:** `q2_customers.csv`

The notebook includes:
- Data preparation and feature scaling
- Elbow method for choosing K
- K-Means clustering
- Cluster centroid interpretation
- PCA dimensionality reduction
- PC1 vs PC2 cluster visualisation

## 1. Data Preparation

In [1]:
# Import required libraries
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [2]:
# Load the dataset
df = pd.read_csv("q2_customers.csv")

print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'q2_customers.csv'

In [ ]:
# Scale all features using StandardScaler
features = df.columns.tolist()

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])

scaled_df = pd.DataFrame(scaled_data, columns=features)

print("Scaled data preview:")
scaled_df.head()

### Why Scaling is Essential before K-Means

K-Means uses distance-based calculations to assign observations to clusters. If the variables are on different scales, features with larger numerical ranges, such as `annual_spend`, can dominate the distance calculation and reduce the importance of smaller-range features like `visits_per_month` or `basket_size`.

Therefore, `StandardScaler` is used to convert all features to a common scale with mean 0 and standard deviation 1. This allows each feature to contribute fairly to the clustering process.

## 2. Choosing K — Elbow Method

In [ ]:
# Compute WCSS for K = 1 to 10
wcss = []

k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_df)
    wcss.append(kmeans.inertia_)

wcss_df = pd.DataFrame({
    "K": list(k_values),
    "WCSS": wcss
})

wcss_df

In [ ]:
# Plot the elbow curve
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), wcss, marker="o")
plt.title("Elbow Method for Choosing Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Within-Cluster Sum of Squares (WCSS)")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

### Elbow Method Interpretation

The WCSS value decreases as K increases because more clusters reduce the distance between data points and their assigned centroids. The optimal K is selected at the point where the decrease in WCSS starts becoming smaller, forming an “elbow”.

Based on the elbow curve, **K = 4** is selected because it provides a good balance between compact clusters and avoiding unnecessary over-segmentation. After K = 4, the improvement in WCSS becomes comparatively smaller.

## 3. K-Means Clustering

In [ ]:
# Fit K-Means with chosen K
chosen_k = 4

kmeans = KMeans(n_clusters=chosen_k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(scaled_df)

print("Cluster counts:")
print(df["cluster"].value_counts().sort_index())

df.head()

In [ ]:
# Cluster centroids in original feature scale for readable interpretation
centroids_scaled = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)

centroids_df = pd.DataFrame(centroids_original, columns=features)
centroids_df.index.name = "cluster"

print("Cluster Centroids in Original Scale:")
centroids_df.round(2)

In [ ]:
# Cluster summary using mean values
cluster_summary = df.groupby("cluster")[features].mean().round(2)
cluster_summary

### Business Interpretation of Clusters

The cluster centroids and summary table can be used to describe each customer segment in business terms:

- **Cluster 0:** Customers in this segment can be interpreted based on whether their spending, visits, and basket size are above or below the overall average. If they show high visits and moderate spend, they may represent loyal routine shoppers.
- **Cluster 1:** If this cluster has high annual spending and larger basket size, it represents high-value customers who should be targeted with premium offers and loyalty benefits.
- **Cluster 2:** If this group has low visits, low annual spend, and high days since last visit, it may represent inactive or at-risk customers who need reactivation campaigns.
- **Cluster 3:** If this group has frequent visits but lower basket size, it may represent regular low-spending customers who can be encouraged through bundle offers or cross-selling.

The exact final interpretation should be read from the centroid table above, because the cluster labels are assigned automatically by K-Means.

## 4. Dimensionality Reduction with PCA

In [ ]:
# Apply PCA to reduce scaled data to 2 principal components
pca = PCA(n_components=2, random_state=42)
pca_data = pca.fit_transform(scaled_df)

pca_df = pd.DataFrame(pca_data, columns=["PC1", "PC2"])
pca_df["cluster"] = df["cluster"]

print("Explained Variance Ratio:")
for i, ratio in enumerate(pca.explained_variance_ratio_, start=1):
    print(f"PC{i}: {ratio:.4f}")

pca_df.head()

In [ ]:
# Feature loadings for PC1 and PC2
loadings_df = pd.DataFrame(
    pca.components_,
    columns=features,
    index=["PC1", "PC2"]
)

print("PCA Feature Loadings:")
loadings_df.round(4)

### PCA Interpretation

The PCA loadings show how strongly each original feature contributes to PC1 and PC2.

- **PC1** captures the direction with the highest variation in the customer data. Features with large positive or negative loadings on PC1 are the main drivers of separation along the horizontal axis.
- **PC2** captures the second-highest variation, independent of PC1. Features with large loadings on PC2 explain the vertical separation between customers.

For example, if `annual_spend`, `basket_size`, and `num_categories_purchased` have strong loadings on PC1, then PC1 may represent overall customer value. If `visits_per_month` and `days_since_last_visit` load strongly in opposite directions on PC2, then PC2 may represent customer engagement or recency.

## 5. Cluster Visualisation

In [ ]:
# Scatter plot of PC1 vs PC2 coloured by cluster label
plt.figure(figsize=(8, 6))

for cluster in sorted(pca_df["cluster"].unique()):
    cluster_points = pca_df[pca_df["cluster"] == cluster]
    plt.scatter(
        cluster_points["PC1"],
        cluster_points["PC2"],
        label=f"Cluster {cluster}",
        alpha=0.75
    )

plt.title("Customer Segments Visualised using PCA")
plt.xlabel("Principal Component 1 (PC1)")
plt.ylabel("Principal Component 2 (PC2)")
plt.legend(title="Cluster")
plt.grid(True)
plt.show()

## Final Summary

This notebook completed the full unsupervised learning workflow:

1. Loaded the customer dataset.
2. Scaled all features using `StandardScaler`.
3. Used the elbow method to compare WCSS for K = 1 to 10.
4. Selected K = 4 based on the elbow point.
5. Fitted K-Means clustering and added cluster labels to the dataframe.
6. Printed readable cluster centroids and interpreted the clusters in business terms.
7. Applied PCA to reduce the dataset to 2 components.
8. Printed explained variance ratios and feature loadings.
9. Visualised clusters using a PC1 vs PC2 scatter plot.